# Clasificación RNN de calidad del agua

Dataset sintético multivariable inspirado en el flujo de trabajo de `example_RNN.ipynb`. La RNN observa 30 días y clasifica la calidad del agua del día siguiente.

## 1. Factores considerados

Se utilizan ocho indicadores: pH, temperatura, turbidez, oxígeno disuelto, conductividad, nitratos, fosfatos y coliformes. Los datos incluyen estacionalidad, dependencia temporal, ruido y episodios de contaminación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

In [ ]:
n_dias = 12000
t = np.arange(n_dias)
fechas = pd.date_range("1994-01-01", periods=n_dias, freq="D")
estacion = np.sin(2 * np.pi * t / 365.25)

# Estado latente de contaminación con memoria y eventos ocasionales.
contaminacion = np.zeros(n_dias, dtype=np.float32)
eventos = rng.random(n_dias) < 0.018
for i in range(1, n_dias):
    impacto = rng.uniform(0.25, 0.75) if eventos[i] else 0.0
    contaminacion[i] = np.clip(
        0.92 * contaminacion[i - 1] + impacto + rng.normal(0, 0.035), 0, 1
    )

# Las variables responden de forma coherente a contaminación y estación.
ph = 7.2 - 0.9 * contaminacion + rng.normal(0, 0.16, n_dias)
temperatura_c = 21 + 6 * estacion + 2 * contaminacion + rng.normal(0, 0.8, n_dias)
turbidez_ntu = 1.5 + 15 * contaminacion + rng.gamma(1.5, 0.7, n_dias)
oxigeno_disuelto_mg_l = 8.5 - 4.2 * contaminacion - 0.07 * (temperatura_c - 20) + rng.normal(0, 0.3, n_dias)
conductividad_us_cm = 280 + 620 * contaminacion + rng.normal(0, 35, n_dias)
nitratos_mg_l = 1.2 + 9 * contaminacion + rng.gamma(1.3, 0.35, n_dias)
fosfatos_mg_l = 0.08 + 1.4 * contaminacion + rng.gamma(1.2, 0.05, n_dias)
coliformes_ufc_100ml = 8 + 1800 * contaminacion ** 2 + rng.lognormal(1.2, 0.8, n_dias)

df_agua = pd.DataFrame({
    "fecha": fechas, "ph": ph, "temperatura_c": temperatura_c,
    "turbidez_ntu": turbidez_ntu,
    "oxigeno_disuelto_mg_l": oxigeno_disuelto_mg_l,
    "conductividad_us_cm": conductividad_us_cm,
    "nitratos_mg_l": nitratos_mg_l, "fosfatos_mg_l": fosfatos_mg_l,
    "coliformes_ufc_100ml": coliformes_ufc_100ml
})
df_agua.head()

## 2. Puntaje y clase de calidad

Se construye un índice sintético de 0 a 100. Las penalizaciones aumentan cuando los indicadores se alejan de condiciones deseables. Las clases son: **Mala** (<50), **Aceptable** (50–74.99) y **Buena** (≥75).

In [ ]:
penalizaciones = np.column_stack([
    np.clip(np.abs(ph - 7.2) / 1.5, 0, 1),
    np.clip(np.abs(temperatura_c - 22) / 12, 0, 1),
    np.clip(turbidez_ntu / 18, 0, 1),
    np.clip((8.5 - oxigeno_disuelto_mg_l) / 5, 0, 1),
    np.clip((conductividad_us_cm - 250) / 800, 0, 1),
    np.clip(nitratos_mg_l / 12, 0, 1),
    np.clip(fosfatos_mg_l / 1.8, 0, 1),
    np.clip(np.log1p(coliformes_ufc_100ml) / np.log1p(2000), 0, 1)
])
pesos = np.array([0.10, 0.05, 0.16, 0.18, 0.09, 0.14, 0.10, 0.18])
df_agua["indice_calidad"] = np.clip(100 * (1 - penalizaciones @ pesos), 0, 100)
df_agua["clase_calidad"] = pd.cut(
    df_agua["indice_calidad"], bins=[-np.inf, 50, 75, np.inf],
    labels=[0, 1, 2], right=False
).astype(int)

nombres_clase = {0: "Mala", 1: "Aceptable", 2: "Buena"}
df_agua["calidad"] = df_agua["clase_calidad"].map(nombres_clase)
print(df_agua["calidad"].value_counts())
df_agua.head()

In [ ]:
features = [
    "ph", "temperatura_c", "turbidez_ntu", "oxigeno_disuelto_mg_l",
    "conductividad_us_cm", "nitratos_mg_l", "fosfatos_mg_l",
    "coliformes_ufc_100ml"
]

fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for ax, feature in zip(axes.flat, features):
    ax.plot(df_agua["fecha"].iloc[:730], df_agua[feature].iloc[:730], linewidth=0.8)
    ax.set_title(feature)
axes.flat[-1].bar(df_agua["calidad"].value_counts().index, df_agua["calidad"].value_counts().values)
axes.flat[-1].set_title("Balance de clases")
plt.suptitle("Primeros dos años de indicadores sintéticos", fontsize=15)
plt.tight_layout(); plt.show()

## 3. División, escala y secuencias

La división es cronológica 70/15/15. El mínimo y máximo se calculan solo con entrenamiento para evitar fuga de información. Cada muestra tendrá forma `[30 días, 8 features]`.

In [ ]:
n_steps = 30
valores = df_agua[features].to_numpy(dtype=np.float32)
etiquetas = df_agua["clase_calidad"].to_numpy(dtype=np.int64)
n_total = len(df_agua)
fin_train = int(n_total * 0.70)
fin_valid = int(n_total * 0.85)

min_train = valores[:fin_train].min(axis=0)
max_train = valores[:fin_train].max(axis=0)
rango_train = np.where(max_train > min_train, max_train - min_train, 1.0)
valores_norm = ((valores - min_train) / rango_train).astype(np.float32)

X = np.lib.stride_tricks.sliding_window_view(
    valores_norm, window_shape=n_steps, axis=0
).transpose(0, 2, 1)[:-1].copy()
y = etiquetas[n_steps:].copy()
indices_y = np.arange(n_steps, n_total)

mask_train = indices_y < fin_train
mask_valid = (indices_y >= fin_train) & (indices_y < fin_valid)
mask_test = indices_y >= fin_valid
X_train, y_train = X[mask_train], y[mask_train]
X_valid, y_valid = X[mask_valid], y[mask_valid]
X_test, y_test = X[mask_test], y[mask_test]

print("Train:", X_train.shape, y_train.shape)
print("Validación:", X_valid.shape, y_valid.shape)
print("Test:", X_test.shape, y_test.shape)
print("Rango normalizado en train:", X_train.min().round(3), X_train.max().round(3))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class WaterQualityDataset(Dataset):
  def __init__(self, X, y):
    self.X, self.y = X, y
  def __len__(self):
    return len(self.X)
  def __getitem__(self, ix):
    return torch.from_numpy(self.X[ix]), torch.tensor(self.y[ix], dtype=torch.long)

dataset = {
    "train": WaterQualityDataset(X_train, y_train),
    "eval": WaterQualityDataset(X_valid, y_valid),
    "test": WaterQualityDataset(X_test, y_test)
}
dataloader = {
    "train": DataLoader(dataset["train"], batch_size=64, shuffle=True),
    "eval": DataLoader(dataset["eval"], batch_size=64, shuffle=False),
    "test": DataLoader(dataset["test"], batch_size=64, shuffle=False)
}

## 4. Clasificador RNN

In [ ]:
class WaterQualityRNN(torch.nn.Module):
  def __init__(self, input_size=8, hidden_size=32, n_clases=3):
    super().__init__()
    self.rnn = torch.nn.RNN(input_size, hidden_size, num_layers=2, batch_first=True)
    self.fc = torch.nn.Linear(hidden_size, n_clases)
  def forward(self, x):
    x, h = self.rnn(x)
    return self.fc(x[:, -1, :])

model = WaterQualityRNN()
model

In [ ]:
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

def fit(model, dataloader, epochs=30, patience=5):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()
    best_loss, best_state, wait = np.inf, None, 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in tqdm(range(1, epochs + 1)):
        model.train(); losses = []
        for X_batch, y_batch in dataloader["train"]:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward(); optimizer.step(); losses.append(loss.item())

        model.eval(); val_losses, correctos, total = [], 0, 0
        with torch.no_grad():
            for X_batch, y_batch in dataloader["eval"]:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                val_losses.append(criterion(logits, y_batch).item())
                correctos += (logits.argmax(1) == y_batch).sum().item()
                total += len(y_batch)

        train_loss, val_loss = np.mean(losses), np.mean(val_losses)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(correctos / total)
        print(f"Época {epoch:02d}: loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={correctos/total:.3f}")

        if val_loss < best_loss - 1e-4:
            best_loss, wait = val_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping"); break

    model.load_state_dict(best_state); model.to(device)
    return history

torch.manual_seed(SEED)
history = fit(model, dataloader)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval(); y_true, y_pred = [], []
with torch.no_grad():
    for X_batch, y_batch in dataloader["test"]:
        pred = model(X_batch.to(device)).argmax(1).cpu().numpy()
        y_pred.extend(pred); y_true.extend(y_batch.numpy())

etiquetas_clase = ["Mala", "Aceptable", "Buena"]
print(classification_report(y_true, y_pred, target_names=etiquetas_clase, digits=3))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=etiquetas_clase, yticklabels=etiquetas_clase)
plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title("Matriz de confusión")
plt.show()

## 5. Clasificar una muestra

Una muestra para esta RNN debe contener 30 días y los ocho factores en el mismo orden. Se normaliza con los parámetros de entrenamiento antes de entrar al modelo.

In [ ]:
def clasificar_muestra(model, muestra_30_dias):
    muestra = np.asarray(muestra_30_dias, dtype=np.float32)
    if muestra.shape != (n_steps, len(features)):
        raise ValueError(f"Se esperaba forma ({n_steps}, {len(features)}), se recibió {muestra.shape}")
    muestra_norm = ((muestra - min_train) / rango_train).astype(np.float32)
    entrada = torch.from_numpy(muestra_norm).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probabilidades = torch.softmax(model(entrada), dim=1).cpu().numpy()[0]
    clase = int(probabilidades.argmax())
    return nombres_clase[clase], probabilidades

# Ejemplo: usar los últimos 30 días disponibles.
muestra_ejemplo = df_agua[features].iloc[-n_steps:].to_numpy(dtype=np.float32)
clase, probabilidades = clasificar_muestra(model, muestra_ejemplo)
print("Calidad predicha:", clase)
print(dict(zip(["Mala", "Aceptable", "Buena"], probabilidades.round(3))))